In [ ]:
# Cell 1 — locate saved , copy source, patch known NUTS bugs, and preflight.
from pathlib import Path
from datetime import datetime, timezone
import json, shutil, subprocess, sys, uuid
import numpy as np

INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working')

# Select the latest save summary with all three simulation families present.
summaries = []
for p in INPUT.rglob('day7_physics_summary.json'):
    try:
        d = json.loads(p.read_text(encoding='utf-8'))
        # Day 7 writes simulations_by_family; accept simulation_counts too for older variants.
        counts = d.get('simulation_counts', d.get('simulations_by_family', {}))
        def draw_count(value):
            if isinstance(value, dict):
                return int(value.get('unique_parameter_draws', 0))
            return int(value or 0)
        if all(draw_count(counts.get(k, 0)) >= 100_000 for k in ('binary', 'white_dwarf', 'zeeman')):
            summaries.append((p, d))
    except Exception as exc:
        print('Skipping unreadable summary:', p, exc)
if not summaries:
    raise FileNotFoundError('Attach the saved Day 7 Physics output containing day7_physics_summary.json and its simulation artifacts.')
SUMMARY_PATH, DAY7 = sorted(summaries, key=lambda item: item[1].get('created_utc', ''), reverse=True)[0]
print('[PASS] Day 7 summary:', SUMMARY_PATH)
print('[PASS] Simulated parameter draws:', DAY7.get('simulation_draws_total'))

# Find saved known-truth examples within the attached output.
expected_run_name = Path(DAY7['artifact_root']).name
validation_dirs = [p for p in INPUT.rglob('nuts_validation_inputs')
                   if p.is_dir() and p.parent.name == expected_run_name
                   and all((p / f'{name}.csv').is_file() for name in ('binary', 'white_dwarf', 'zeeman'))]
if not validation_dirs:
    raise FileNotFoundError(f'Could not find saved Day 7 NUTS validation inputs for {expected_run_name}. Reattach the saved Day 7 output.')
VALIDATION_DIR = validation_dirs[0]
print('[PASS] Reusing synthetic validation inputs:', VALIDATION_DIR)

# Find the project package in the attached Cosmic Swan 4 source.
project_candidates = []
for pp in INPUT.rglob('pyproject.toml'):
    root = pp.parent
    if (root / 'src/cosmic_black_swan/inference.py').is_file() and (root / 'src/cosmic_black_swan/physics.py').is_file():
        project_candidates.append(root)
if not project_candidates:
    raise FileNotFoundError('Attach Cosmic Swan 4, with cbs-project-src/pyproject.toml and src/cosmic_black_swan/inference.py.')
PROJECT_SOURCE = next((p for p in project_candidates if 'cosmic-swan-4' in str(p).lower()), project_candidates[0])
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:6]
RUN_ROOT = WORK / f'day7-nuts-recovery-{RUN_ID}'
PROJECT = RUN_ROOT / 'cbs-project-src'
RUN_ROOT.mkdir(parents=True, exist_ok=False)
shutil.copytree(PROJECT_SOURCE, PROJECT)

# Fix BLS initialization: trial durations must be shorter than minimum_period=0.1 d.
inference_file = PROJECT / 'src/cosmic_black_swan/inference.py'
source = inference_file.read_text(encoding='utf-8')
old_duration = 'durations = np.geomspace(0.005, min(1.5, maximum_period * 0.12), 16)'
new_duration = 'durations = np.geomspace(0.005, min(0.09, maximum_period * 0.12), 16)'
if old_duration not in source:
    raise RuntimeError('Expected BLS duration line was not found; stop and inspect the attached source version.')
source = source.replace(old_duration, new_duration, 1)

# Fix Zeeman tracing: wavelength bounds must be computed as Python constants before JAX tracing.
old_sig = 'def _zeeman_model(wavelength, observed_flux, flux_error, field_max_gauss: float):'
new_sig = 'def _zeeman_model(wavelength, observed_flux, flux_error, field_max_gauss: float, wavelength_min: float, wavelength_max: float):'
old_center = 'center = numpyro.sample("line_center_angstrom", dist.Uniform(float(wavelength.min()), float(wavelength.max())))'
new_center = 'center = numpyro.sample("line_center_angstrom", dist.Uniform(wavelength_min, wavelength_max))'
old_wave = """        wave_j, flux_j, error_j = map(jnp.asarray, (wave, observed, errors))

        def model():
            _zeeman_model(wave_j, flux_j, error_j, field_max_gauss)"""
new_wave = """        wavelength_min, wavelength_max = float(np.min(wave)), float(np.max(wave))
        wave_j, flux_j, error_j = map(jnp.asarray, (wave, observed, errors))

        def model():
            _zeeman_model(wave_j, flux_j, error_j, field_max_gauss, wavelength_min, wavelength_max)"""
for old, label in ((old_sig, 'Zeeman model signature'), (old_center, 'Zeeman wavelength bounds'), (old_wave, 'Zeeman model call')):
    if old not in source:
        raise RuntimeError(f'Expected source text not found: {label}; stop and inspect the attached source version.')
source = source.replace(old_sig, new_sig, 1).replace(old_center, new_center, 1).replace(old_wave, new_wave, 1)
inference_file.write_text(source, encoding='utf-8')

PACKAGE_SRC = PROJECT / 'src'
sys.path.insert(0, str(PACKAGE_SRC))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(PROJECT)], check=True)
try:
    import numpyro, jax, torch
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numpyro>=0.15'], check=True)
    import numpyro, jax, torch
if not torch.cuda.is_available() or jax.default_backend() != 'gpu':
    raise RuntimeError(f'Expected Kaggle GPU runtime; torch_cuda={torch.cuda.is_available()}, JAX backend={jax.default_backend()}')
from cosmic_black_swan.inference import run_numpyro_nuts
print('[PASS] Patched package import:', inference_file)
print('[PASS] PyTorch GPU:', torch.cuda.get_device_name(0))
print('[PASS] JAX devices:', jax.devices())
print('[PASS] Working disk free GiB:', round(shutil.disk_usage(WORK).free / (1024**3), 2))
print('Recovery output:', RUN_ROOT)

In [ ]:
# Cell 2 — retry only binary-star and Zeeman NUTS; white-dwarf NUTS.
STAGE_RESULTS = []
for idx, family in enumerate(('binary', 'zeeman')):
    print(f'\n===== NUTS recovery: {family} =====', flush=True)
    try:
        result = run_numpyro_nuts(
            family=family,
            input_path=VALIDATION_DIR / f'{family}.csv',
            output_dir=RUN_ROOT / 'physics_posteriors',
            warmup=500, samples=1000, chains=4, seed=720 + idx,
            field_max_gauss=20_000.0,
        )
        row = {'family': family, 'status': 'complete', 'result': result}
        print(json.dumps(row, indent=2, default=str), flush=True)
    except Exception as exc:
        row = {'family': family, 'status': 'failed', 'error': f'{type(exc).__name__}: {exc}'}
        print(json.dumps(row, indent=2), flush=True)
    STAGE_RESULTS.append(row)
    (RUN_ROOT / 'nuts_recovery_stage_log.json').write_text(json.dumps(STAGE_RESULTS, indent=2, default=str), encoding='utf-8')
print('Retry stages finished. A printed complete status is required for each family; inspect diagnostics, too.')

In [ ]:
# Cell 3 — combine retry results with the white-dwarf result and write a compact report.
nuts_root = RUN_ROOT / 'physics_posteriors'
results = {}
for row in STAGE_RESULTS:
    family = row['family']
    path = nuts_root / family / 'nuts_run.json'
    if path.is_file():
        saved = json.loads(path.read_text(encoding='utf-8'))
        results[family] = {
            'retry_status': row['status'],
            'sampling_diagnostics': saved.get('sampling_diagnostics'),
            'posterior_summary': saved.get('posterior_summary'),
            'model_assumptions': saved.get('model_assumptions'),
        }
    else:
        results[family] = {'retry_status': row['status'], 'error': row.get('error', 'No nuts_run.json created')}
results['white_dwarf_previous_day7_result'] = DAY7.get('numpyro_nuts_results', {}).get('white_dwarf', {'status': 'not found in Day 7 summary'})
report = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'day7_input_summary': str(SUMMARY_PATH),
    'source_project': str(PROJECT_SOURCE),
    'recovery_output_root': str(RUN_ROOT),
    'fixes_applied': [
        'BLS initialization durations capped below the 0.1-day minimum trial period.',
        'Zeeman wavelength bounds computed from NumPy arrays before JAX model tracing.'
    ],
    'nuts_results': results,
    'scope_limits': [
        'All inference examples use known-truth synthetic inputs, not real TESS stars or observed spectra.',
        'The Day 7 conditional flow results remain undercovered on held-out simulations and require redesign or calibration.',
        'Toy binary and weak-field Zeeman assumptions omit important astrophysical/instrumental effects.'
    ]
}
report_path = WORK / 'day7_1_nuts_recovery_summary.json'
report_path.write_text(json.dumps(report, indent=2, default=str), encoding='utf-8')
print('Recovery report:', report_path)
print(json.dumps({k: {'retry_status': v.get('retry_status'), 'diagnostics': v.get('sampling_diagnostics')} if isinstance(v, dict) else v for k, v in results.items()}, indent=2, default=str))
print('All saved files under:', RUN_ROOT)